In [49]:
import json
from argparse import Namespace

import matplotlib
import numpy as np
import yaml
from IPython.display import HTML
from matplotlib import animation
from matplotlib import pyplot as plt
from tqdm import tqdm

from collab_env.data.file_utils import expand_path, get_project_root
from collab_env.gnn.gnn_3D.analyze_results import (
    analyze_results,
    convert_attention_weights_to_adj_matrix,
    load_attention_weights,
    process_training_result,
)
from collab_env.gnn.gnn_3D.build_dataset import Sim3DInMemoryDataset
from collab_env.gnn.gnn_3D.train_3DGNN import model_factory, train_3DGNN
from collab_env.sim.boids.run_simulator import run_simulator_main

In [ ]:
%matplotlib inline

## Generate the simulation data

Use the simulator to generate training data. 

In [50]:
config_file = expand_path("docs/gnn/gnn3D/config.yaml", get_project_root())
print(config_file)

run_folder = run_simulator_main(
    config_file=config_file
)  # "/Users/tc/ArchivedBoxSync/Research/Basis/collab-env-stuff/collab-3D-GNN/collab-environment/collab_env/sim/boids/config.yaml")

/Users/tc/ArchivedBoxSync/Research/Basis/collab-env-stuff/collab-3D-GNN/collab-environment/docs/gnn/gnn3D/config.yaml


  0%|          | 0/5 [00:00<?, ?it/s]/Users/tc/ArchivedBoxSync/Research/Basis/collab-env-stuff/collab-3D-GNN/.venv/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:142: UserWarning: WARN: The obs returned by the `reset()` method was expecting a tuple, actual type: <class 'list'>
  logger.warn(f"{pre} was expecting a tuple, actual type: {type(obs)}")
/Users/tc/ArchivedBoxSync/Research/Basis/collab-env-stuff/collab-3D-GNN/.venv/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:134: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be float32, actual type: float64
  logger.warn(
/Users/tc/ArchivedBoxSync/Research/Basis/collab-env-stuff/collab-3D-GNN/.venv/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/Users/tc/ArchivedBoxSync/

In [51]:
print("run folder ", run_folder)

run folder  /Users/tc/ArchivedBoxSync/Research/Basis/collab-env-stuff/collab-3D-GNN/collab-environment/sim-output/sim_gnn3D_demo-started-20260116-173340


## Build the GNN dataset

Build the dataset using the simulation data created above. You can specify the columns to include as node features using the node_feature_columns parameter. This will create a processed folder inside the run_folder. The processed folder will include pytorch files containing the sequence of graphs for each episode. 

In [52]:
node_feature_columns = "distance_to_target_mesh_closest_point_1,target_mesh_closest_point_x,target_mesh_closest_point_y,target_mesh_closest_point_z,mesh_scene_distance,mesh_scene_closest_point_x,mesh_scene_closest_point_y,mesh_scene_closest_point_z"
dataset = Sim3DInMemoryDataset(
    run_folder,
    label_type="velocities",
    time_window_length=1,
    node_feature_columns=node_feature_columns.split(","),
)

Processing...
100%|██████████| 5/5 [00:01<00:00,  3.54it/s]
Done!


loading 5 episodes


100%|██████████| 5/5 [00:00<00:00, 15.45it/s]


Display the dataset metadata just to make sure things look ok. 

In [53]:
with open(dataset.processed_paths[0], "r") as f:
    dataset_metadata = json.load(f)
print(dataset_metadata)

{'node_feature_columns': ['distance_to_target_mesh_closest_point_1', 'target_mesh_closest_point_x', 'target_mesh_closest_point_y', 'target_mesh_closest_point_z', 'mesh_scene_distance', 'mesh_scene_closest_point_x', 'mesh_scene_closest_point_y', 'mesh_scene_closest_point_z'], 'time_window_length': 1, 'input_node_dim': 8, 'edge_attr_dim': 3, 'label_dim': 3, 'episode_file_list': ['episode-0-completed-20260116-173357.parquet', 'episode-1-completed-20260116-173413.parquet', 'episode-2-completed-20260116-173430.parquet', 'episode-3-completed-20260116-173447.parquet', 'episode-4-completed-20260116-173504.parquet'], 'label_type': 'velocities'}


## Train the GNN 
Train a GNN on the dataset created above. The GNN model is created by the model_factory() function. If you want a different model, replace this function with a function that constructs your model. 

In [54]:
training_result_path = expand_path("training_results", run_folder)
training_result_path.mkdir(parents=True, exist_ok=True)
print("training result path: ", training_result_path)

training result path:  /Users/tc/ArchivedBoxSync/Research/Basis/collab-env-stuff/collab-3D-GNN/collab-environment/sim-output/sim_gnn3D_demo-started-20260116-173340/training_results


In [55]:
result = train_3DGNN(
    directory=run_folder,
    training_result_path=training_result_path,
    num_epochs=1,
    evaluate_only=False,
    batch_size=1,
    model_creation_function=model_factory(),
)

loading 5 episodes


100%|██████████| 5/5 [00:00<00:00, 15.15it/s]


val:   0%|          | 0/1 [00:00<?, ?episode/s]

train:   0%|          | 0/4 [00:00<?, ?episode/s]

val:   0%|          | 0/1 [00:00<?, ?episode/s]

Store the results in the trainin_results subfolder. 

In [ ]:
process_training_result(result, "sim-output/" + run_folder.name + "/training_results")

## Analyze Results

### Rollouts

For each episode in the simulation data, run the model in evaluation mode starting at the initial position of the episode. The position predicted at time $t$ will be the input position for time $t+1$.

First we create the agent config dictionary indicating the model to load and run. 


In [ ]:
agent_config = dict()
agent_config["start_time"] = 0
agent_config["dataset_metadata_file"] = "sim-output/" + run_folder.name + "/processed/gnn3D_data"  # type: ignore [assignment]
agent_config["model_file"] = (
    "sim-output/"
    + run_folder.name  # type: ignore[assignment]
    + "/training_results/saved_models/gnn-Attention-Linear_epoch_0.pt"
)

Next create the args for the analyze_results() function. 

In [ ]:
args = Namespace()
args.directory = "sim-output/" + run_folder.name
args.show_visualizer = False
args.rollout = True
args.rollout_subdirectory = "rollout"
args.agent_config_file = "gnn_agent_config.yaml"
args.simulator_config_file = "config.yaml"
args.plot_attention = False
args.animate_attention = False
args.predictions_are_velocities = True

Perform rollouts on all episodes. This will show the predicted positions of the agents over time in the Open3D simulator starting from the initial position in the episode. You can hit Q in the window to quit the individual rollouts. This model hasn't learned much, so the rollouts will be bad -- if you don't see any boids, zoom all the way out and you may get a glimpse of these very shy creatures, kind of like when I went to the Denver Zoo and couldn't find the red panda. (That was the same trip where the gorilla tried to put his fist through the plexiglass into my face.)  

There are a lot of warnings from the simulator that need to be fixed at some point because they are annoying. 

In [ ]:
agent_config_path = expand_path("gnn_agent_config.yaml", run_folder)
episose_list = run_folder.glob("*episode*.parquet")
for episode in tqdm(episose_list):
    agent_config["position_file"] = "sim-output/" + run_folder.name + "/" + episode.name
    yaml.dump(agent_config, open(agent_config_path, "w"))
    analyze_results(args)

### Animate attention weights

Display the attention weights as they change through the time. 

First set up the args for animate_attention(). Make sure you set the filename to one of the attention weight files that were generated by the training step above. These will be in the training_results subdirectory. 

In [ ]:
args = Namespace()

args.directory = "sim-output/" + run_folder.name
args.show_visualizer = True
args.rollout = False
args.rollout_subdirectory = "rollout"
args.agent_config_file = "gnn_agent_config.yaml"
args.simulator_config_file = "config.yaml"
args.plot_attention = False
args.animate_attention = True
args.predictions_are_velocities = True


validation_attention_list = list(
    training_result_path.glob("validation_attention_weights*.parquet")
)
print("val att list ", validation_attention_list)

args.filename = "training_results/" + validation_attention_list[0].name
print("attention weights file", args.filename)

Next, run the matplotlib animation.

In [ ]:
"""
Args:
    attention_weights_list (list): list of attention weights of shape (time, num agents, num agents)
"""
print(
    "This takes a little over a minute to load on my laptop. There is a play button at the bottom. If you get a file not found error, make sure the args.filename above is set correctly. That name changes every time you train. "
)
matplotlib.rcParams["animation.embed_limit"] = 100

attention_weights_list = load_attention_weights(args.directory, args.filename)
attention_matrices = [
    convert_attention_weights_to_adj_matrix(w) for w in attention_weights_list
]
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(attention_matrices[0], cmap="viridis", vmin=0.0, vmax=1.0, aspect="auto")
title = ax.set_title("Frame 0")  # , animated=False)

nrows = len(attention_matrices[0])
ncols = len(attention_matrices[0][0])

# Place ticks centered on each pixel/index
ax.set_xticks(np.arange(ncols))
ax.set_yticks(np.arange(nrows))

# Label ticks with integers 0..n-1
ax.set_xticklabels(np.arange(ncols))
ax.set_yticklabels(np.arange(nrows))


def init():
    im.set_data(attention_matrices[0])
    title.set_text("Frame 0")
    return im, title


def update(frame):
    im.set_data(attention_matrices[frame])
    title.set_text(f"Frame: {frame:10.0f}")
    return (im, title)


anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(attention_matrices),
    init_func=init,
    interval=100,
    blit=False,
)

matplotlib.rcParams["animation.html"] = "jshtml"

HTML(anim.to_jshtml())